# Visual Asset Auditing System - Test Bench

This notebook implements the **Test Bench** tier of the Visual Asset Auditing System. It demonstrates:

1. Connecting to AlloyDB and GCS.
2. Running a Hybrid Search (pgvector + FTS).
3. Detecting the drop-off point and selecting the top 60 candidates (High Confidence + Borderline).
4. Running Gemini 3.5 Flash online inference to audit the selected assets.
5. Creating and updating the `audit_results` table in AlloyDB.

*Transcribed from IMG_9422.jpeg. Additional source screenshots will be added below in the order received.*

In [ ]:
# Install required libraries
!pip install -q google-genai google-cloud-vision google-cloud-aiplatform kfp google-cloud-pipeline-components pgvector asyncpg kneed pandas numpy pillow nest-asyncio sqlalchemy "protobuf<5.0.0dev"


## Package Installation

This cell installs all necessary external libraries (Google GenAI, Cloud Vision, AI Platform, pgvector, asyncio) to equip the notebook environment.

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

import google.genai as genai
from google.genai import types

print("Google GenAI SDK imported successfully.")


## Google Cloud Authentication

This cell handles authentication with Google Cloud using Colab auth utilities, sets up API project client.

In [ ]:
# AlloyDB Connection Setup using SQLAlchemy Pool + AsyncConnector
import asyncio
import asyncpg
from typing import Tuple
from sqlalchemy.ext.asyncio import create_async_engine, AsyncEngine
from google.cloud.alloydb.connector import IPTypes, AsyncConnector

_engine_cache = {}
_connector_cache = {}

async def get_alloydb_connection(reuse: bool = True) -> Tuple[AsyncEngine, AsyncConnector]:
    """Establishes and pools AlloyDB connections using SQLAlchemy and the AsyncConnector, with automatic timeout diagnostics."""
    global _engine_cache
    global _connector_cache

    if reuse and 'default' in _engine_cache:
        return _engine_cache['default'], _connector_cache['default']

    # Use lazy refresh for serverless/Colab environments
    connector = AsyncConnector(refresh_strategy="lazy")

    async def getconn():
        # Handle case where user pasted the full resource path or just the instance ID
        instance_uri = ALLOYDB_INSTANCE
        if not instance_uri.startswith("projects/"):
            instance_uri = f"projects/{PROJECT_ID}/locations/{REGION}/clusters/{ALLOYDB_CLUSTER}/instances/{ALLOYDB_INSTANCE}"

        try:
            # Enforce 10-second connection timeout to prevent hanging loop CancelledErrors
            conn = await asyncio.wait_for(
                connector.connect(
                    instance_uri,
                    "asyncpg",
                    user=DB_USER,
                    password=DB_PASSWORD,
                    db=DB_NAME,
                    enable_iam_auth=False, # Set to True if using IAM auth
                    ip_type=IPTypes.PUBLIC # Adjust to PUBLIC or PRIVATE
                ),
                timeout=10.0
            )
            return conn
        except asyncio.TimeoutError:
            raise ConnectionError(
                f"AlloyDB connection timed out (10s) to {instance_uri}. "
                "Ensure your client IP is authorized in the AlloyDB Public IP console, "
                "or check your VPC network access if running internally."
            )
        except Exception as e:
            raise ConnectionError(f"Failed to connect to AlloyDB: {e}")

    engine = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        echo=False,
        pool_size=10,
        max_overflow=20,
        pool_pre_ping=True # Force SQLAlchemy to health-check connections
    )

    if reuse:
        _engine_cache['default'] = engine
        _connector_cache['default'] = connector

    return engine, connector


In [ ]:
# # Run the setup
# import nest_asyncio
# nest_asyncio.apply()
# try:
#     asyncio.run(setup_database())
# except Exception as e:
#     print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# Database Connection Verification (Read-Only Check – No DDL or Schema Changes)
import asyncio
import nest_asyncio

async def verify_database_connection():
    """Verifies connection to AlloyDB and confirms the visual_assets table is reachable without making any DDL changes."""
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            count = await db.fetchval(f"SELECT COUNT(*) FROM {DB_SCHEMA}.visual_assets;")
            print(f"Connected to AlloyDB successfully. Schema '{DB_SCHEMA}.visual_assets' is ready ({count} assets found).")
    except Exception as e:
        print(f"Database connection status: {e}")

# Run connection check
nest_asyncio.apply()
try:
    asyncio.run(verify_database_connection())
except Exception as e:
    print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# 1. Audit Config Generator & Embedding Generation
import json
import asyncio
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List, Dict, Tuple
from pydantic import BaseModel, Field

# Shared executor for query-side embedding calls (avoids spinning up a new pool per request)
_EMBED_EXECUTOR = ThreadPoolExecutor(max_workers=8)

# Define Pydantic model for the structured Audit Context.
class AuditContextModel(BaseModel):
    audit_goal: str = Field(description="Refined, precise version of the user's goal.")
    image_description: Optional[str] = Field(None, description="Description of the reference image if provided, else null.")
    reference_is_composite_canvas: bool = Field(description="True if the reference image is a composite layout, webpage screenshot, hero banner, or real-world photograph containing the logo/asset. False if the reference image represents an isolated, standalone logo on a plain background.")
    inclusion_criteria: List[str] = Field(description="List of 3-7 specific, testable criteria an image MUST meet to be relevant.")
    exclusion_criteria: List[str] = Field(description="List of 2-5 criteria that EXCLUDE an image from relevance.")
    adjudication_logic: str = Field(description="A clear IF-THEN-ELSE statement defining PASS/FAIL conditions.")
    verification_steps: List[str] = Field(description="Ordered list of 5-9 concrete, criteria-derived visual verification steps the auditing vision-LLM must execute verbatim on every candidate image (full-canvas region sweep including tiny thumbnails/favicons/watermarks, shape & geometry, typography & exact spelling, color & gradient treatment, person identity when applicable, final adjudication).")
    search_keywords: List[str] = Field(description="List of 5-10 single-word search terms (e.g. ['woman', 'female', 'portrait']) rather than multi-word phrases, to ensure broad keyword match capability in indexed assets.")
    vision_tag_filter: List[str] = Field(description="List of 2-5 keywords specifically mapped to Google Cloud Vision API tag vocabulary.")
    audit_instructions: str = Field(description="Detailed instructions to be passed to the auditing LLM describing the criteria.")
    extraction_schema: Dict[str, str] = Field(description="Dynamic key-value pairs representing additional boolean/integer/string properties to extract from the image to verify the audit criteria.")

def get_image_mime_type(path: str) -> str:
    """Helper to dynamically resolve visual asset MIME type based on file path extension."""
    lower_path = path.lower()
    if lower_path.endswith(".png"):
        return "image/png"
    elif lower_path.endswith(".webp"):
        return "image/webp"
    elif lower_path.endswith(".gif"):
        return "image/gif"
    return "image/jpeg"

def generate_audit_config(user_goal: str, reference_image_description: Optional[str] = None, available_tags: Optional[List[str]] = None) -> dict:
    """Translates a high-level user goal into a structured audit context with strict visual grounding and anti-hallucination guardrails."""
    image_context = ""
    if reference_image_description:
        image_context = f"\nReference Image Description: {reference_image_description}\n"

    active_tags = available_tags if available_tags else WEB_AUDIT_VISION_TAGS
    tags_pool_str = ", ".join([f"'{t}'" for t in active_tags])

    builder_prompt = f"""
You are a Lead Enterprise Visual & UI/UX Asset Auditor building a high-precision audit configuration for executive leadership.
Your task is to expand a user's audit goal into an airtight, zero-mistake structured audit context.

User Goal: {user_goal}
{image_context}

BRAND COMPLIANCE SIGNATURES INFERENCE:
Analyze the User Goal and the Reference Image Description (if provided) to extract:
1. The target brand, UI component, or visual subject under audit (e.g. Google Pay, YouTube, a specific Favicon, a cookies banner, or Google Workspace logos).
2. What constitutes the COMPLIANT (active, modern, approved) visual design, layout, typography, or shape.
3. What constitutes the NON-COMPLIANT (legacy, outdated, spoofed, or incorrect) visual design, layout, typography, or shape.
Ground all inclusion and exclusion criteria strictly in these inferred compliance signatures.

STEP 1: DYNAMIC BRAND & INTENT EXTRACTION
Identify the target brand/visual subject under audit based on the brand compliance signatures inference. Restrict all criteria, search keywords, and instructions strictly to this extracted subject.

STRICT VISUAL GROUNDING & ANTI-HALLUCINATION (CRITICAL):
- You MUST base all inclusion/exclusion criteria and visual descriptions strictly and exclusively on what is physically visible inside the provided reference image.
- Do NOT assume, extrapolate, or hallucinate the presence of brand names, wordmarks, UI elements, button texts, or logos that are cropped out or missing from the reference image.
- If a button, text, or logo is not visible in the reference image, do NOT include it as a mandatory requirement (inclusion criteria).

STEP 2: COMPLIANCE STATE ALIGNMENT & TEMPLATE ROLE ANALYSIS
Analyze the role of the reference image template (if provided) using your inferred compliance signatures.
- **State Conflict (Negation/Comparative Match)**: If the reference image represents the *Active/Compliant/New* standard, but the user wants to find *outdated/old* assets.
  * The template role is **Negative / Comparative Match**.
  * The exclusion criteria MUST exclude the reference image's compliant visual signatures.
  * The inclusion criteria MUST target older legacy styles of that same brand.
- **State Alignment (Positive Template Match)**: If the reference image represents the *Outdated/Legacy/Old* standard, and the user wants to find *outdated/old* assets.
  * The template role is **Positive Template Match**.
  * The inclusion criteria MUST require matching the reference image's legacy visual signatures.
  * The exclusion criteria MUST explicitly exclude the modern, compliant standard.
- **General Discovery Match**: If the user wants to find *all* assets of the brand regardless of state.
  * The inclusion criteria should pass the reference style AND other iterations of the brand.

STEP 3: REFERENCE COMPOSITE CANVAS DETECTION
Evaluate the description of the reference image:
- Set `reference_is_composite_canvas` to True if it describes a composite scene, real-world photograph, or webpage screenshot containing the logo/asset.
- Set `reference_is_composite_canvas` to False only if it represents an isolated, standalone logo on a plain background.

STEP 4: STRICT BRAND EXCLUSION & ANTI-SPOOFING GUARDRAIL (CRITICAL)
If the target visual subject is a specific sub-brand or product logo:
- You MUST explicitly include in the `exclusion_criteria` a rule to exclude the generic corporate master logo unless it is explicitly accompanied by the sub-brand.
- ANTI-SPOOFING: Add explicit rules to reject look-alike misspellings or related but incorrect sub-brands.

STEP 5: COLOR & CANVAS CONTEXT INDEPENDENCE (CRITICAL)
Unless the user's goal explicitly specifies a color constraint, you MUST explicitly write in the `audit_instructions` and `adjudication_logic` that color is not a match determinant.

STEP 6: STEP-BY-STEP VERIFICATION PLAN (verification_steps) (CRITICAL):
Produce an ordered `verification_steps` list (5-9 steps) that the downstream auditing vision-LLM will execute verbatim on every candidate image. Derive each step from the criteria above. The plan MUST include, in this order:
1. FULL-CANVAS REGION SWEEP: inspect every region of the candidate image (all four corners, header, footer, navigation, buttons, background, center) and enumerate EVERY logo/brand/graphic element found — explicitly including tiny thumbnails, favicons, app icons, watermarks, and partially occluded or low-resolution marks. A valid target match ANYWHERE on the canvas counts, no matter how small.
2. SHAPE & GEOMETRY: verify the structural shape/geometry of each detected target element against the compliance signatures (e.g. interlocking loops vs wordmark, border radius, proportions).
3. TYPOGRAPHY & EXACT SPELLING: verify wordmarks letter-by-letter; reject lookalikes and misspellings.
4. COLOR & GRADIENT: verify the gradient/solid color treatment of each detected element, applying the color-independence rule unless the goal explicitly constrains color.
5. PERSON IDENTITY (only if the audit subject involves a person): verify that the SAME person from the reference image appears (facial features, appearance) — no lookalikes.
6. FINAL ADJUDICATION: apply `adjudication_logic` to the collected evidence.

CRITICAL RULE FOR vision_tag_filter:
You MUST ONLY select tags from the following list that exist in our database:
[{tags_pool_str}]

CRITICAL RULE FOR search_keywords (CRITICAL):
- The search_keywords MUST be a list of single-word search terms rather than multi-word phrases.
- RECALL MANDATE: include the brand's short forms, abbreviations, and common filename tokens (e.g. 'gpay' as well as 'pay', 'yt' as well as 'youtube') so keyword search over filenames and descriptions can catch assets whose text descriptions are sparse.

CRITICAL MANDATES FOR OPTICAL RESOLUTION & CLARITY AUDITS (CRITICAL):
1. **Strict Adjudication Logic**: Write a crystal-clear IF-THEN-ELSE statement in `adjudication_logic`.
   - Recognize Cropped/Blurred Targets: State explicitly that if the target visual asset is cropped, low-resolution, or blurred, but you can still identify its core structural signatures, it remains eligible.
   - Flag Resolution Status: In such cases, you must mark `optical_resolution_sufficient = False` in the extraction schema.
   - Only evaluate `matches_criteria = False` if the image is so extremely degraded, pixelated, or tiny (sub-pixel) that it is mathematically impossible to distinguish it from a generic shape.
2. **Diagnostic Extraction Schema**: In `extraction_schema`, define:
   - `detected_asset_style`: string (Exact description of what is seen on canvas)
   - `is_outdated_or_noncompliant`: boolean
   - `is_embedded_in_composite_hero`: boolean
   - `composite_location_notes`: string
   - `optical_resolution_sufficient`: boolean
"""

    # Passing the Pydantic model directly to the SDK
    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=builder_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AuditContextModel,
            temperature=0.0
        )
    )
    return json.loads(response.text)

def build_fused_query_text(context_dict: dict, is_negative: bool = False) -> str:
    """Combines all context fields into a single rich text representation for embedding."""
    if is_negative:
        return f"Exclude images that: {'; '.join(context_dict.get('exclusion_criteria', []))}. Specifically exclude spoofed misspellings, lookalike brands, and other products."

    parts = [
        f"Audit goal: {context_dict.get('audit_goal')}",
        f"Include images that: {'; '.join(context_dict.get('inclusion_criteria', []))}",
        "IMPORTANT FOR EMBEDDING SIMILARITY: Ignore foreground and background color differences. Focus purely on shape, text layout, logo structural design, and semantic meaning."
    ]
    if context_dict.get("image_description"):
        parts.append(f"Reference image: {context_dict.get('image_description')}")
    return " | ".join(parts)

def _embed_single(contents) -> List[float]:
    """One unambiguous embed_content call -> plain python vector."""
    return client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=contents,
        config=types.EmbedContentConfig(output_dimensionality=768)
    ).embeddings[0].values

def _l2_normalize(vec) -> List[float]:
    v = np.asarray(vec, dtype=np.float32)
    n = np.linalg.norm(v)
    return (v / n).tolist() if n > 0 else v.tolist()

async def embed_query_probes(context_dict: dict, reference_image_path: Optional[str] = None) -> Dict[str, List[float]]:
    """Generates ALL query-side probe vectors concurrently for multi-probe retrieval:
       - 'text'     : positive fused audit text probe
       - 'negative' : exclusion-criteria text probe (contrastive demotion)
       - 'image'    : reference image probe (skipped gracefully if the embed model rejects images)
       - 'fused'    : L2-normalized average of image+text probes (multimodal probe)
    Each probe is embedded in its OWN call so image and text signals never mask each other."""
    pos_text = build_fused_query_text(context_dict, is_negative=False)[:1500]
    neg_text = build_fused_query_text(context_dict, is_negative=True)[:1500]

    loop = asyncio.get_running_loop()
    text_future = loop.run_in_executor(_EMBED_EXECUTOR, _embed_single, pos_text)
    neg_future = loop.run_in_executor(_EMBED_EXECUTOR, _embed_single, neg_text)

    image_future = None
    if reference_image_path:
        mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            img_part = types.Part.from_uri(file_uri=reference_image_path, mime_type=mime)
        else:
            with open(reference_image_path, "rb") as f:
                img_part = types.Part.from_bytes(data=f.read(), mime_type=mime)
        image_future = loop.run_in_executor(_EMBED_EXECUTOR, _embed_single, [img_part])

    probes = {}
    probes["text"], probes["negative"] = await asyncio.gather(text_future, neg_future)
    if image_future is not None:
        try:
            probes["image"] = list(await image_future)
            probes["fused"] = _l2_normalize(
                np.asarray(_l2_normalize(probes["image"]), dtype=np.float32) + np.asarray(_l2_normalize(probes["text"]), dtype=np.float32)
            )
        except Exception as e:
            print(f"Warning: reference-image embedding probe unavailable ({e}). Falling back to text-only retrieval probes.")
    return probes

async def embed_audit_context(context_dict: dict, reference_image_path: Optional[str] = None) -> Tuple[List[float], List[float]]:
    """Backwards-compatible wrapper returning a (positive, negative) embedding pair."""
    probes = await embed_query_probes(context_dict, reference_image_path)
    pos_vec = probes.get("fused") or probes.get("image") or probes["text"]
    return pos_vec, probes["negative"]

In [ ]:
# 2. Parallel Hybrid Search with RRF & Drop-Off Detection
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
import json
import asyncio
from kneed import KneeLocator
from pgvector.asyncpg import register_vector
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel, Field

# ---- Retrieval tunables (recall vs latency) ----
VECTOR_ARM_LIMIT = 1500          # per-probe ANN candidates (index-friendly, evaluated per emitted row)
TEXT_ARM_LIMIT = 4000            # FTS / tag arm row caps
CONTRASTIVE_NEG_WEIGHT = 0.3     # client-side demotion weight for negative-probe similarity
VECTOR_SAFEGUARD_DIST = 0.28     # absolute visual-similarity rescue threshold
CROSS_ENCODER_SKIP_N = 60        # at/below this candidate count, skip text reranking entirely
CROSS_ENCODER_BATCH = 25         # candidates scored per single LLM call

class _BatchScoreItem(BaseModel):
    index: int = Field(description="The candidate index exactly as given in the input list.")
    score: int = Field(description="Relevance score 0-100 for that candidate.")

class _BatchScores(BaseModel):
    scores: List[_BatchScoreItem] = Field(description="One score entry for EVERY candidate index in the input.")

async def run_semantic_reranking_and_filter(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_context: dict, max_workers: int = 25) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies LLM Cross-Encoder semantic scoring in BATCHED calls (~25 candidates per call instead of
    1 call per candidate), with visual-vector and keyword-promotion safeguards to protect recall.
    Small candidate sets skip this stage entirely — the vision inference stage is the real judge."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")

    # Fast path: for small sets a text-only filter can only add latency and lose recall.
    if len(candidates) <= CROSS_ENCODER_SKIP_N:
        print(f"[Cross-Encoder] {len(candidates)} candidates <= {CROSS_ENCODER_SKIP_N}: skipping text reranking (vision inference will adjudicate all of them).")
        return df_high.reset_index(drop=True), df_edge.reset_index(drop=True), pd.DataFrame()

    audit_goal = audit_context.get("audit_goal", "")
    inclusion_str = "; ".join(audit_context.get("inclusion_criteria", []))

    # 1. Visual Vector Safeguard (absolute visual distance check) — bypasses text scoring entirely.
    prescored, to_score = [], []
    for row in candidates:
        if row.get("vector_distance", 1.0) < VECTOR_SAFEGUARD_DIST:
            prescored.append({
                **row,
                "cross_encoder_score": 100,  # Max score to force-promote
                "relevance_score": row.get("relevance_score", 0.0) * 2.0,  # Visual match boost
                "vector_safeguard_triggered": True
            })
        else:
            to_score.append(row)

    def _score_batch(batch: list) -> list:
        lines = []
        for j, row in enumerate(batch):
            desc = str(row.get("gemini_description") or "")[:400]
            tags = ", ".join([str(t) for t in (row.get("vision_tags") or [])])[:200]
            fname = str(row.get("asset_filename") or "")[:120]
            lines.append(f"[{j}] filename: {fname} | tags: {tags} | description: {desc}")
        joined = "\n".join(lines)

        prompt = f"""You are a rapid relevance scoring engine. Score EVERY candidate below for relevance to the audit goal.
Audit Goal: {audit_goal}
Inclusion Criteria: {inclusion_str}

Score each candidate's textual relevance 0-100 using this calibrated rubric:
- 80-100 (High): Direct matches to the target subject, clear presence of target visual elements, or standalone target brand logos.
- 30-79 (Borderline): Contextual matches, related terms/brands, composite graphics/screenshots/hero banners that may CONTAIN the target somewhere (even small), or descriptions with visual layout complexity.
- 0-29 (Low): Clearly unrelated subjects (different products, unrelated graphics).

IMPORTANT RECALL RULE: A sparse, generic, or missing description is NOT evidence of irrelevance — the target may appear as a small element the description skipped. Score such candidates as Borderline (30-79), never Low.

Candidates:
{joined}

Return one score entry for every index 0..{len(batch) - 1}."""
        try:
            resp = client.models.generate_content(
                model=GEMINI_CROSS_ENCODER_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=_BatchScores,
                    temperature=0.0
                )
            )
            parsed = {int(s["index"]): int(s["score"]) for s in json.loads(resp.text)["scores"]}
        except Exception as e:
            print(f"[Cross-Encoder] Batch scoring failed ({e}); defaulting batch to neutral 50.")
            parsed = {}
        return [parsed.get(j, 50) for j in range(len(batch))]

    batches = [to_score[i:i + CROSS_ENCODER_BATCH] for i in range(0, len(to_score), CROSS_ENCODER_BATCH)]
    print(f"[Cross-Encoder] Semantic reranking of {len(to_score)} candidates in {len(batches)} batched LLM calls (+{len(prescored)} vector-safeguard promotions)...")

    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        batch_scores = await asyncio.gather(*[loop.run_in_executor(executor, _score_batch, b) for b in batches])

    reranked = list(prescored)
    for batch, scores in zip(batches, batch_scores):
        for row, score in zip(batch, scores):
            ce_mult = max(0.1, score / 50.0)
            reranked.append({
                **row,
                "cross_encoder_score": score,
                "relevance_score": row.get("relevance_score", 0.0) * ce_mult,
                "vector_safeguard_triggered": False
            })

    high_list, edge_list, low_list = [], [], []
    for item in reranked:
        if item.get("vector_safeguard_triggered", False):
            filename = item.get("asset_filename") or str(item.get("gcs_raw_path", "")).split("/")[-1]
            dist = item.get("vector_distance", 0)
            print(f"🛡️ Vector Safeguard triggered: Force-promoted {str(filename)[:40]} due to high visual similarity (distance: {dist:.4f})")
            high_list.append(item)
            continue

        score = item["cross_encoder_score"]
        if score >= 75:
            high_list.append(item)
        elif score >= 30:
            edge_list.append(item)
        elif item.get("promoted_by_keyword_or_tag", False):
            # Recall safeguard: exact keyword/tag hits are never text-filtered out of the audit.
            item["cross_encoder_score"] = 30
            edge_list.append(item)
        else:
            low_list.append(item)

    df_high_final = pd.DataFrame(high_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if high_list else pd.DataFrame()
    df_edge_final = pd.DataFrame(edge_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if edge_list else pd.DataFrame()
    df_low_final = pd.DataFrame(low_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if low_list else pd.DataFrame()

    return df_high_final, df_edge_final, df_low_final

def compute_weighted_rrf_rerank(candidates: list, audit_context: dict) -> list:
    """Zero-latency in-memory multi-factor reranker. Executes in local CPU RAM (< 0.5ms) without external API overhead."""
    tag_filter = [t.lower().strip() for t in audit_context.get("vision_tag_filter", []) if t]
    search_keywords = [k.lower().strip() for k in audit_context.get("search_keywords", []) if k]
    reranked = []

    for item in candidates:
        row = item["data"]
        base_score = item["score"]
        multiplier = 1.0

        # 1. Vision tag exact hit boost (2.0x per matching tag up to 8x)
        tags = [str(t).lower() for t in (row.get("vision_tags") or [])]
        tag_hits = sum(1 for t in tags if any(ft in t or t in ft for ft in tag_filter))
        if tag_hits > 0:
            multiplier *= (2.0 ** min(tag_hits, 3))

        # 2. Keyword exact hit in filename or description boost (1.5x per matching keyword up to 2.25x)
        desc = str(row.get("gemini_description") or "").lower()
        fname = str(row.get("asset_filename") or "").lower()
        kw_hits = sum(1 for kw in search_keywords if kw in desc or kw in fname)
        if kw_hits > 0:
            multiplier *= min(1.5 ** kw_hits, 2.25)

        reranked.append({
            "data": row,
            "score": base_score * multiplier,
            "base_rrf_score": base_score,
            "rerank_multiplier": multiplier,
            "vector_distance": item.get("vector_distance", 1.0)
        })

    reranked.sort(key=lambda x: x["score"], reverse=True)
    return reranked

async def run_hybrid_search(scope_config: dict, audit_context: dict, reference_image_path: Optional[str] = None, limit: int = 10000) -> List[dict]:
    """Performs parallel MULTI-PROBE Vector + Keyword/Filename + Tag search with RRF fusion, local reranking,
    and deduplication (No silent cliff).

    Latency: every vector arm orders by a bare `embedding <=> $probe` so the ANN index (HNSW/ScaNN) is used;
    the contrastive negative-probe demotion is applied client-side on the retrieved rows only.
    Recall: up to three vector probes (fused image+text, image-only, text-only) give diluted composite
    embeddings (tiny thumbnails inside screenshots) multiple chances to surface, while the FTS arm also
    matches brand tokens inside asset filenames."""
    probes = await embed_query_probes(audit_context, reference_image_path)

    engine, _ = await get_alloydb_connection()

    search_keywords = [k for k in audit_context.get("search_keywords", []) if k]
    keyword_query_str = " OR ".join(search_keywords) if search_keywords else ""
    fname_patterns = [f"%{k.lower().strip()}%" for k in search_keywords]
    tag_patterns = [f"%{t.lower().strip()}%" for t in audit_context.get("vision_tag_filter", []) if t]

    SELECT_COLS = "asset_id, gcs_raw_path, format, vision_tags, gemini_description, asset_filename, page_url, content_hash"

    async def _tune_ann_recall(db):
        # Best-effort ANN recall bump; unsupported GUCs are ignored silently.
        for guc in ("SET hnsw.ef_search = 400", "SET ivfflat.probes = 20", "SET scann.num_leaves_to_search = 200"):
            try:
                await db.execute(guc)
            except Exception:
                pass

    # Run the vector probe arms + keyword arm + tag arm concurrently on pooled connections.
    async def run_vector(arm_name: str, probe_vec):
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            await register_vector(db)
            await _tune_ann_recall(db)
            rows = await db.fetch(
                f"""
                SELECT {SELECT_COLS},
                       (embedding <=> $1::vector) AS vector_distance,
                       (embedding <=> $2::vector) AS neg_distance
                FROM {DB_SCHEMA}.visual_assets
                ORDER BY embedding <=> $1::vector
                LIMIT $3
                """,
                probe_vec, probes["negative"], min(limit, VECTOR_ARM_LIMIT)
            )
            out = [dict(r) for r in rows]
            # Contrastive demotion (pos_dist - w * neg_dist) computed in-memory, keeping the ANN index usable.
            out.sort(key=lambda r: float(r["vector_distance"]) - CONTRASTIVE_NEG_WEIGHT * float(r["neg_distance"]))
            return arm_name, out

    async def run_fts():
        if not keyword_query_str:
            return "fts", []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT {SELECT_COLS},
                       ts_rank_cd(to_tsvector('english', coalesce(gemini_description, '')),
                                  websearch_to_tsquery('english', $1)) AS fts_rank
                FROM {DB_SCHEMA}.visual_assets
                WHERE to_tsvector('english', coalesce(gemini_description, '')) @@ websearch_to_tsquery('english', $1)
                   OR asset_filename ILIKE ANY($2::text[])
                ORDER BY fts_rank DESC
                LIMIT $3
                """,
                keyword_query_str, fname_patterns, min(limit, TEXT_ARM_LIMIT)
            )
            return "fts", [dict(r) for r in rows]

    async def run_tags():
        if not tag_patterns:
            return "tags", []
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT {SELECT_COLS}
                FROM {DB_SCHEMA}.visual_assets
                WHERE EXISTS (
                    SELECT 1 FROM unnest(vision_tags) tag
                    WHERE tag ILIKE ANY($1::text[])
                )
                LIMIT $2
                """,
                tag_patterns, min(limit, TEXT_ARM_LIMIT)
            )
            return "tags", [dict(r) for r in rows]

    vector_arms = []
    if "fused" in probes:
        vector_arms.append(("vec_fused", probes["fused"]))
    if "image" in probes:
        vector_arms.append(("vec_image", probes["image"]))
    vector_arms.append(("vec_text", probes["text"]))

    arm_tasks = [run_vector(name, vec) for name, vec in vector_arms] + [run_fts(), run_tags()]
    arm_results = dict(await asyncio.gather(*arm_tasks))

    fts_results = arm_results.get("fts", [])
    tag_results = arm_results.get("tags", [])

    promoted_ids = set()
    for r in fts_results[:200]:
        promoted_ids.add(str(r["asset_id"]))
    for r in tag_results[:200]:
        promoted_ids.add(str(r["asset_id"]))

    # Best (minimum) vector distance per asset across all probe arms.
    vector_distance_map = {}
    for name, _ in vector_arms:
        for r in arm_results[name]:
            aid = str(r["asset_id"])
            d = float(r["vector_distance"])
            if aid not in vector_distance_map or d < vector_distance_map[aid]:
                vector_distance_map[aid] = d

    # Multi-Arm RRF Fusion (k=60). Total vector weight 0.60 split evenly across active probes.
    k = 60
    results_map = {}
    per_vec_weight = 0.60 / len(vector_arms)

    def upsert_ranks(results_list, weight=1.0):
        for rank, row in enumerate(results_list):
            img_id = str(row["asset_id"])
            if img_id not in results_map:
                clean = {c: v for c, v in row.items() if c not in ("vector_distance", "neg_distance", "fts_rank")}
                results_map[img_id] = {"data": clean, "score": 0.0, "vector_distance": vector_distance_map.get(img_id, 1.0)}
            results_map[img_id]["score"] += weight / (k + rank + 1)

    for name, _ in vector_arms:
        upsert_ranks(arm_results[name], weight=per_vec_weight)
    if fts_results:
        upsert_ranks(fts_results, weight=0.25)
    if tag_results:
        upsert_ranks(tag_results, weight=0.15)

    fused = list(results_map.values())

    # Zero-Latency In-Memory Reranking (Provides a smooth score curve for Kneedle)
    reranked_fused = compute_weighted_rrf_rerank(fused, audit_context)

    # Deduplication
    seen_identifiers = set()
    deduplicated = []
    for item in reranked_fused:
        row = item["data"]
        img_id = str(row["asset_id"])
        img_identifier = row.get("content_hash") or row.get("gcs_raw_path")
        if img_identifier not in seen_identifiers:
            seen_identifiers.add(img_identifier)
            is_promoted = img_id in promoted_ids
            deduplicated.append({
                **row,
                "relevance_score": item["score"],
                "base_rrf_score": item.get("base_rrf_score", 0),
                "rerank_multiplier": item.get("rerank_multiplier", 1),
                "promoted_by_keyword_or_tag": is_promoted,
                "vector_distance": item.get("vector_distance", 1.0)
            })

    return deduplicated[:limit]

def detect_dropoff_flawless(df: pd.DataFrame, sensitivity: float = 1.0) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies Kneedle curvature + rolling volatility to segment candidates into High, Edge, and Low."""
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_sorted = df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)
    y = df_sorted["relevance_score"].values
    x = np.arange(len(y))

    y_min, y_max = y.min(), y.max()
    if y_max == y_min:
        return df_sorted.iloc[:int(len(y)*0.3)], df_sorted.iloc[int(len(y)*0.3):int(len(y)*0.6)], df_sorted.iloc[int(len(y)*0.6):]

    y_norm = (y - y_min) / (y_max - y_min + 1e-9)
    x_norm = x / (len(x) - 1)

    coords = np.column_stack((x_norm, y_norm))
    line_start, line_end = coords[0], coords[-1]
    line_vec = line_end - line_start
    line_vec_norm = line_vec / np.sqrt(np.sum(line_vec**2))
    vec_from_start = coords - line_start
    scalar_proj = np.dot(vec_from_start, line_vec_norm)
    proj_on_line = np.outer(scalar_proj, line_vec_norm)
    dist_to_line = np.sqrt(np.sum((coords - proj_on_line)**2, axis=1))

    idx1 = np.argmax(dist_to_line)

    window = max(3, int(len(y) * 0.05))
    rolling_std = pd.Series(y_norm).rolling(window=window, center=True).std().fillna(0).values
    noise_threshold = np.mean(rolling_std) * (0.6 / sensitivity)

    idx2 = len(y) - 1
    for i in range(idx1 + 2, len(rolling_std)):
        if rolling_std[i] < noise_threshold:
            idx2 = i
            break

    min_borderline_width = max(10, int((len(y) - idx1) * 0.25))
    if (idx2 - idx1) < min_borderline_width:
        idx2 = min(len(y) - 1, idx1 + min_borderline_width)

    idx1 = max(10, idx1)

    high_df = df_sorted.iloc[:idx1 + 1].copy()
    edge_df = df_sorted.iloc[idx1 + 1: idx2 + 1].copy()
    low_df = df_sorted.iloc[idx2 + 1:].copy()

    # Visual Vector Safeguard: Check if any candidate has extremely high similarity (distance < 0.28)
    # even if it is currently classified in Low_df (or has been discarded).
    # Force rescue these to protect visual recall.
    if "vector_distance" in low_df.columns:
        rescued_vec = low_df[low_df["vector_distance"] < VECTOR_SAFEGUARD_DIST].copy()
        if not rescued_vec.empty:
            edge_df = pd.concat([edge_df, rescued_vec], ignore_index=True)
            low_df = low_df[low_df["vector_distance"] >= VECTOR_SAFEGUARD_DIST].copy()
            print(f"🛡️ Vector Safeguard triggered in Kneedle: Force-rescued {len(rescued_vec)} candidate(s) from Low to Borderline based on high visual similarity.")

    if "promoted_by_keyword_or_tag" in low_df.columns:
        rescued = low_df[low_df["promoted_by_keyword_or_tag"] == True].copy()
        if not rescued.empty:
            edge_df = pd.concat([edge_df, rescued], ignore_index=True)
            low_df = low_df[low_df["promoted_by_keyword_or_tag"] != True].copy()
            print(f"🛡️ Safeguard triggered: Promoted {len(rescued)} composite/diluted candidates from Low to Borderline tier based on exact keyword/tag match.")

    return high_df, edge_df, low_df

## Retrieval, Reranking, and Drop-off Segmentation Engine

This cell defines the core search engine:

- `run_hybrid_search`: Combines Vector, Full-Text, and Tag search arms into a fused RRF list, applying zero-latency keyword boosts.
- `detect_dropoff_flawless`: Curvature-based Kneedle algorithm that truncates irrelevant tail results.

In [ ]:
# 3. Hydration & Parallel LLM Audit Inference
import asyncio
from concurrent.futures import ThreadPoolExecutor
import json
import pandas as pd
from typing import List, Tuple, Optional
from google.genai import types
from google.cloud import storage
from pydantic import create_model, Field
import time
from PIL import Image
import io

# One cached GCS client — creating a client per download costs an auth/channel round-trip every time.
_GCS_CLIENT = None
def _get_storage_client():
    global _GCS_CLIENT
    if _GCS_CLIENT is None:
        _GCS_CLIENT = storage.Client(project=PROJECT_ID)
    return _GCS_CLIENT

# Only truly extreme images are downscaled, so tiny embedded logos/thumbnails stay legible to the model.
MAX_IMAGE_DIM = 3072

# Transient API errors worth retrying (rate limits / server hiccups); everything else fails fast.
_TRANSIENT_ERROR_MARKERS = ("429", "500", "503", "resource_exhausted", "unavailable", "deadline", "internal", "overloaded", "timeout", "timed out")

def _generate_with_retry(make_call, max_retries: int = 3):
    """Retries a Gemini call on transient errors with exponential backoff (1.5s, 3s)."""
    for attempt in range(max_retries):
        try:
            return make_call()
        except Exception as e:
            transient = any(m in str(e).lower() for m in _TRANSIENT_ERROR_MARKERS)
            if attempt == max_retries - 1 or not transient:
                raise
            time.sleep(1.5 * (2 ** attempt))

def process_transparency(image_bytes: bytes, default_bg: Tuple[int, int, int] = (30, 30, 30)) -> bytes:
    """Detects alpha transparency and composites it onto a solid dark background so light elements/text
    remain visible; also caps extreme resolutions (> MAX_IMAGE_DIM px) to cut upload/inference latency."""
    try:
        img = Image.open(io.BytesIO(image_bytes))
        needs_flatten = img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info)
        needs_resize = max(img.size) > MAX_IMAGE_DIM
        if not needs_flatten and not needs_resize:
            return image_bytes
        if needs_flatten:
            img = img.convert('RGBA')
            bg = Image.new("RGBA", img.size, default_bg + (255,))
            img = Image.alpha_composite(bg, img).convert("RGB")
        else:
            img = img.convert("RGB")
        if needs_resize:
            scale = MAX_IMAGE_DIM / float(max(img.size))
            img = img.resize((max(1, int(img.width * scale)), max(1, int(img.height * scale))), Image.LANCZOS)
        out_bytes = io.BytesIO()
        img.save(out_bytes, format="PNG")
        return out_bytes.getvalue()
    except Exception as e:
        print(f"Warning: Failed to preprocess image transparency: {e}")
    return image_bytes

def _prep_image(image_bytes: bytes) -> Tuple[bytes, str]:
    """Returns (bytes, mime) after transparency flattening / extreme-size capping, sniffing the real format."""
    processed = process_transparency(image_bytes)
    if processed is not image_bytes:
        return processed, "image/png"
    try:
        fmt = (Image.open(io.BytesIO(image_bytes)).format or "PNG").lower()
        mime = {"jpeg": "image/jpeg", "png": "image/png", "webp": "image/webp", "gif": "image/gif"}.get(fmt, "image/png")
    except Exception:
        mime = "image/png"
    return image_bytes, mime

def _download_asset_bytes(path: str) -> bytes:
    """Downloads raw bytes from GCS (cached client) or the local filesystem."""
    if path.startswith("gs://"):
        bucket_name = path.split("/")[2]
        blob_name = "/".join(path.split("/")[3:])
        return _get_storage_client().bucket(bucket_name).blob(blob_name).download_as_bytes()
    with open(path, "rb") as f:
        return f.read()

def enforce_zero_false_positives_rules(df: pd.DataFrame) -> pd.DataFrame:
    """Enforces zero-false-positive criteria. Demotes matches if confidence is below 75%."""
    if df.empty:
        return df

    def guardrail_check(row):
        matches = bool(row.get("matches_criteria", False))
        try:
            conf = int(float(row.get("match_confidence", 100)))
        except (TypeError, ValueError):
            conf = 100
        rationale = str(row.get("visual_analysis_step_by_step", "")) + " " + str(row.get("match_rationale", ""))
        if matches and conf < 75:
            row["matches_criteria"] = False
            row["match_rationale"] = f"[GUARDRAIL DEMOTION: Conf {conf}% < 75%] {rationale}"
        return row

    return df.apply(guardrail_check, axis=1)

def build_dynamic_audit_model(extraction_schema: dict):
    """Builds the structured-output Pydantic model ONCE per run (identical for every candidate)."""
    fields = {
        "visual_analysis_step_by_step": (
            str,
            Field(description="CHAIN OF THOUGHT: Execute the numbered VERIFICATION STEPS in order and record dense, factual findings per step BEFORE any conclusion (target under 180 words): every canvas region inspected, EVERY brand/logo/graphic element found (including tiny thumbnails, favicons, watermarks), and the shape / typography / color-gradient findings for each.")
        ),
        "criteria_verdicts": (
            str,
            Field(description="One line per criterion, in order: 'I1: PASS|FAIL — short evidence' for each inclusion criterion, then 'E1: TRIGGERED|CLEAR — short evidence' for each exclusion criterion.")
        ),
        "matches_criteria": (
            bool,
            Field(description="Strict final evaluation: True ONLY if the asset matches the target criteria and passes adjudication_logic (even inside a composite hero banner). False otherwise.")
        ),
        "match_confidence": (
            int,
            Field(description="Match Confidence percentage (0-100%). You must assign < 75 if there is any doubt or visual occlusion.")
        ),
        "match_rationale": (
            str,
            Field(description="A concise final executive rationale explaining exactly why matches_criteria evaluated to True or False based on the visual_analysis_step_by_step.")
        )
    }

    for field_name, field_info in (extraction_schema or {}).items():
        if field_name in ["matches_criteria", "match_confidence", "match_rationale", "visual_analysis_step_by_step", "criteria_verdicts"]:
            continue

        t = str
        desc = f"Extracted value for {field_name}"

        if isinstance(field_info, dict):
            ftype = field_info.get("field_type", "string").lower()
            desc = field_info.get("description", desc)
        else:
            ftype = str(field_info).lower()

        if ftype == "boolean":
            t = bool
        elif ftype == "integer":
            t = int
        elif ftype == "number":
            t = float

        fields[field_name] = (t, Field(description=desc))

    return create_model("DynamicAuditModel", **fields)

_DEFAULT_VERIFICATION_STEPS = [
    "FULL-CANVAS REGION SWEEP: Systematically inspect every region of the candidate image (all four corners, header, footer, navigation, buttons, background, center) and enumerate EVERY logo, brand mark, icon, or graphic element found — explicitly including tiny thumbnails, favicons, app badges, watermarks, and partially occluded or low-resolution marks.",
    "SHAPE & GEOMETRY: For each detected element, verify its structural shape and geometry against the target's compliance signatures.",
    "TYPOGRAPHY & EXACT SPELLING: Verify any wordmarks letter-by-letter; reject lookalike or misspelled branding.",
    "COLOR & GRADIENT: Note the color/gradient treatment of each detected element; apply the Color Independence Rule unless the criteria explicitly constrain color.",
    "PERSON IDENTITY (if applicable): If the audit subject involves a specific person, verify the SAME person appears (facial features, appearance) — no lookalikes.",
    "FINAL ADJUDICATION: Apply the Strict Adjudication Rule to the evidence collected above."
]

def build_audit_prompt(audit_config: dict, has_reference_image: bool) -> str:
    """Builds the (identical for every candidate) audit prompt ONCE per run, wiring the config's
    step-by-step verification plan and the reference-image comparison directives into the LLM call."""
    audit_instructions = audit_config.get("audit_instructions", "")
    inclusion_criteria = audit_config.get("inclusion_criteria", [])
    exclusion_criteria = audit_config.get("exclusion_criteria", [])
    adjudication_logic = audit_config.get("adjudication_logic", "")
    is_composite = bool(audit_config.get("reference_is_composite_canvas", False))

    inclusion_str = "\n".join([f"- I{i+1}: {c}" for i, c in enumerate(inclusion_criteria)]) if inclusion_criteria else "- None specified"
    exclusion_str = "\n".join([f"- E{i+1}: {c}" for i, c in enumerate(exclusion_criteria)]) if exclusion_criteria else "- None specified"

    verification_steps = audit_config.get("verification_steps") or _DEFAULT_VERIFICATION_STEPS
    steps_str = "\n".join([f"{i+1}. {s}" for i, s in enumerate(verification_steps)])

    reference_instructions = ""
    if has_reference_image:
        if is_composite:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Target Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE (CRITICAL):
The reference image (image_0) is a **Composite Canvas** (a complex real-world photograph/screenshot containing the logo).
Do NOT expect the candidate image under audit (image_1) to contain the hands, terminals, backgrounds, or full layout seen in image_0.
Instead, look at the target logo/brand style (e.g. logos or wordmark shown on the phone screen) inside image_0.
Verify if the candidate image (image_1) contains that target logo style. Ignore all other background visual noise in image_0.
"""
        else:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Image Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE:
Since the Reference Image (image_0) is a standalone logo, compare the candidate image (image_1) side-by-side against image_0.
The candidate matches even if the target appears as a SMALL element inside a larger composite/screenshot — a match anywhere on the canvas counts.
"""

    return f"""
Evaluate this image against the specified audit goal and criteria with high precision.

{reference_instructions}

[Color Independence Rule]:
Unless the Inclusion Criteria explicitly mention a required color, you must ignore any color differences between the reference image and the candidate image.

[Exact Typography Rule]:
Pay strict attention to typography and spelling (e.g. 'Google Play' is NOT 'Google Pay', 'Ads' is NOT 'AdWords'). Reject any assets that contain lookalike or misspelled branding unless the inclusion criteria explicitly permit them.

AUDIT SCOPE & EVALUATION RULES:

[Inclusion Criteria – Asset MUST fulfill these to pass]:
{inclusion_str}

[Exclusion Criteria – If asset triggers any of these, it MUST fail]:
{exclusion_str}

[Strict Adjudication Rule]:
{adjudication_logic}

[General Audit Instructions]:
{audit_instructions}

VERIFICATION STEPS (execute IN ORDER; record findings for each step in `visual_analysis_step_by_step`):
{steps_str}

FINAL EXECUTION:
1. Execute every VERIFICATION STEP above, scanning the entire canvas — a valid target match ANYWHERE on the candidate image counts, even as a small thumbnail inside a composite hero graphic.
2. Extract all diagnostic visual features requested in the schema, including whether the target is embedded inside a composite hero graphic (`is_embedded_in_composite_hero`).
3. Fill `criteria_verdicts` with one line per criterion (I1..In: PASS|FAIL, E1..En: TRIGGERED|CLEAR) citing the visual evidence.
4. Apply the Strict Adjudication Rule to your recorded evidence, then set `matches_criteria` to true only if the inclusion criteria are satisfied without triggering any exclusion criteria.
5. Provide a clear justification in `match_rationale` and an honest `match_confidence` (< 75 whenever occlusion, blur, or tiny scale creates doubt).
"""

async def run_llm_audit_single(asset_data: dict, audit_config: dict, _executor=None, reference_image_part: Optional[types.Part] = None, _model=None, _prompt=None) -> dict:
    """Evaluates a single image asset: hydration (GCS download + transparency blend) and Gemini inference
    are pipelined in one worker task, with transient-error retries. Schema/prompt are prebuilt per run."""
    DynamicAuditModel = _model or build_dynamic_audit_model(audit_config.get("extraction_schema", {}))
    audit_prompt = _prompt or build_audit_prompt(audit_config, reference_image_part is not None)
    gcs_path = asset_data["gcs_raw_path"]

    def _hydrate_and_infer():
        img_bytes, mime = _prep_image(_download_asset_bytes(gcs_path))
        image_part = types.Part.from_bytes(data=img_bytes, mime_type=mime)

        contents = []
        if reference_image_part:
            contents.append(reference_image_part)
        contents.append(image_part)
        contents.append(audit_prompt)

        return _generate_with_retry(lambda: client.models.generate_content(
            model=GEMINI_INFERENCE_MODEL,
            contents=contents,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=DynamicAuditModel,
                temperature=0.0
            )
        ))

    loop = asyncio.get_running_loop()
    try:
        response = await loop.run_in_executor(_executor, _hydrate_and_infer)
        extracted_data = json.loads(response.text)
    except Exception as e:
        extracted_data = {
            "matches_criteria": False,
            "match_confidence": 0,
            "match_rationale": f"Audit evaluation failed due to error: {str(e)}",
            "error": str(e)
        }

    return {**asset_data, **extracted_data}

async def run_llm_inference_on_dropoff_results(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_config: dict, max_workers: int = 30, reference_image_path: Optional[str] = None) -> pd.DataFrame:
    """Runs parallel multi-threaded LLM inference on candidate subsets, supporting side-by-side template
    matches, GCS/local transparency preprocessing, retries, and per-run prompt/schema caching."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")

    print(f"Starting parallel LLM audit inference on {len(candidates)} candidates (Max concurrency: {max_workers})...")

    # Helper to download and preprocess the reference image once
    def _get_reference_part():
        ref_bytes, mime = _prep_image(_download_asset_bytes(reference_image_path))
        return types.Part.from_bytes(data=ref_bytes, mime_type=mime)

    t0 = time.time()
    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        reference_image_part = None
        if reference_image_path:
            reference_image_part = await loop.run_in_executor(executor, _get_reference_part)

        # Build the structured-output schema and audit prompt ONCE for the whole run.
        shared_model = build_dynamic_audit_model(audit_config.get("extraction_schema", {}))
        shared_prompt = build_audit_prompt(audit_config, reference_image_part is not None)

        tasks = [
            run_llm_audit_single(asset, audit_config, _executor=executor, reference_image_part=reference_image_part, _model=shared_model, _prompt=shared_prompt)
            for asset in candidates
        ]
        results = await asyncio.gather(*tasks)

    duration = time.time() - t0
    print(f" Processing assets... [{len(candidates)}/{len(candidates)}] completed in {duration:.1f}s")
    print(f" Inference completed. Enforcing Zero-FP policies and sorting scoreboard...")

    results_df = pd.DataFrame(results)
    results_df = enforce_zero_false_positives_rules(results_df)

    if "relevance_score" in results_df.columns:
        results_df = results_df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)

    print(f"\n=== VERIFIED AUDIT SCOREBOARD (Sorted by Search Similarity) ===")
    for i, r in results_df.iterrows():
        status_label = "PASS" if r.get("matches_criteria", False) else "FAIL"
        filename = r.get("asset_filename") or r.get("gcs_raw_path", "").split("/")[-1]
        sim = r.get("relevance_score", 0.0)
        conf = r.get("match_confidence", 100)
        print(f"[{i+1}/{len(results_df)}] {status_label} | {str(filename)[:40]} | Sim: {sim:.4f} | Conf: {conf}%")

    return results_df

In [ ]:
# 4. Calibration Summary & Audit Results Saving E2E Loop
async def save_audit_results_to_db(session_id: str, results_df: pd.DataFrame):
    """Saves the final audited results into AlloyDB (Bypassed by default in the interactive runner)."""
    if results_df.empty:
        return

    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        records = results_df.to_dict("records")
        for r in records:
            verdict = "PASS" if r.get("matches_criteria") is True else "FAIL"

            # Extract criteria_checks from JSON response or construct it
            criteria_checks = {k: v for k, v in r.items() if k not in ["asset_id", "gcs_raw_path", "matches_criteria", "error", "asset_filename", "page_url"]}

            await db.execute(
                f"""
                INSERT INTO {DB_SCHEMA}.audit_results (
                    session_id, asset_id, overall_verdict, adjudication_result,
                    criteria_checks, rationale, confidence_band
                ) VALUES ($1, $2, $3, $4, $5, $6, $7)
                """,
                session_id,
                r.get("asset_id"),
                verdict,
                r.get("matches_criteria", False),
                json.dumps(criteria_checks),
                r.get("match_rationale", "Completed"),
                "high" if r.get("match_confidence", 0) > 95 else ("borderline" if r.get("match_confidence", 0) >= 70 else "below_threshold")
            )
    print("Audit results saved to database.")

async def generate_ai_audit_summary(results_df: pd.DataFrame, audit_config: dict) -> str:
    """Generates an AI-powered executive summary of the visual asset audit results."""
    import json
    import numpy as np
    if results_df is None or results_df.empty:
        return "No audit results available to generate a summary."

    # Select columns to pass to the LLM, avoiding internal or verbose vector columns
    exclude_cols = {"num_chunks", "max_relevance_score", "gcs_raw_path", "gcs_processed_path", "embedding", "embedding_at"}
    cols_to_include = [col for col in results_df.columns if col not in exclude_cols]

    # Convert to records safely, handling NumPy arrays, lists, and floats without boolean truth value ambiguity
    clean_df = results_df[cols_to_include].copy()
    for col in clean_df.columns:
        def safe_clean(val):
            if val is None:
                return None
            if isinstance(val, (np.ndarray, pd.Series)):
                return val.tolist() if val.size > 0 else None
            if isinstance(val, float) and pd.isna(val):
                return None
            return val
        clean_df[col] = clean_df[col].apply(safe_clean)

    records = clean_df.to_dict(orient="records")
    formatted_results = json.dumps(records, indent=2)

    audit_instructions = audit_config.get("audit_instructions", "No specific audit context provided.")

    # Compute basic stats to seed in the prompt
    total_audited = len(results_df)
    matches_col = "matches_criteria" if "matches_criteria" in results_df.columns else None
    if matches_col:
        # Convert to boolean safely, handling string representation if any
        matches_true = results_df[matches_col].apply(lambda x: str(x).lower() in ("true", "1", "yes")).sum()
    else:
        matches_true = "N/A"

    summary_prompt = f"""
You are a Lead Visual Asset Auditor.
Your task is to write a visually engaging, highly structured, and extremely concise summary of a visual asset audit Test Bench calibration run.

STRICT RULES FOR FORMATTING & SECTIONS:
1. You MUST only include the following exact three sections in the output:
   - Objective & Scope (Preview Subset) (within the top [!NOTE] block)
   - Calibration Statistics (as a numbered list)
   - Configuration Calibration Insights (as a single, brief narrative paragraph of 3-4 sentences detailing the visual rules performance)
2. DO NOT include any other sections.
3. DO NOT pass any definitive verdicts of success.

AUDIT CONTEXT (Visual Evaluation Criteria):
{audit_instructions}

TEST BENCH STATS:
- Total images audited: {total_audited}
- Total images matching criteria (True): {matches_true}

TEST BENCH FINDINGS (JSON):
{formatted_results}
"""

    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=summary_prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text


## Telemetry Summaries & Calibration Database Saving

This cell defines functions to generate final executive AI summaries (`generate_ai_audit_summary`) and save audit session details into the run calibration tables in AlloyDB for telemetry history.

In [ ]:
# 5. E2E Execution Helpers (Interactive Split Stages)
import base64
import os
import time
import json
import pandas as pd
from google.cloud import storage

def get_gcs_image_base64(gcs_path: str) -> str:
    """Downloads image from GCS or local file and returns its base64 data URI for inline HTML rendering."""
    try:
        image_bytes = b""
        mime_type = "image/png"

        # Detect format
        lower_path = gcs_path.lower()
        if lower_path.endswith(".jpg") or lower_path.endswith(".jpeg"):
            mime_type = "image/jpeg"
        elif lower_path.endswith(".webp"):
            mime_type = "image/webp"
        elif lower_path.endswith(".gif"):
            mime_type = "image/gif"

        if gcs_path.startswith("gs://"):
            parts = gcs_path.replace("gs://", "").split("/", 1)
            bucket_name = parts[0]
            blob_name = parts[1]

            # Authenticated download using python client
            storage_client = storage.Client()
            bucket = storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            image_bytes = blob.download_as_bytes()
        elif os.path.exists(gcs_path):
            with open(gcs_path, "rb") as f:
                image_bytes = f.read()
        else:
            return ""

        encoded = base64.b64encode(image_bytes).decode("utf-8")
        return f"data:{mime_type};base64,{encoded}"
    except Exception as e:
        return ""

async def generate_audit_config_only(user_goal: str, reference_image_path: Optional[str] = None) -> dict:
    """Stage 1: Analyzes reference image (Forensic) and generates structured Audit Configuration."""
    reference_image_description = None
    if reference_image_path:
        print("0. Performing forensic executive analysis on reference image...")
        ref_mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            image_part = types.Part.from_uri(file_uri=reference_image_path, mime_type=ref_mime)
        else:
            with open(reference_image_path, "rb") as f:
                image_part = types.Part.from_bytes(data=f.read(), mime_type=ref_mime)

        forensic_analysis_prompt = """You are a Lead Executive Visual & UI/UX Inspector. Perform an exhaustive, forensic-level breakdown of this uploaded reference image for enterprise audit configuration.
Provide a comprehensive, structured analysis starting with Brand Hierarchy:
1. Target Brand & Scope: What exact brand or product is shown? (e.g. specifically Google Pay / G Pay, or Google Workspace). Do not mix in separate products.
2. Primary Asset Hierarchy & Category: Classify whether this image is a Standalone Brand Logo, a UI Component (Payment Button, Cookie Modal), or a Composite Canvas (Hero Banner, Screenshot, Collage).
3. Exact Visual Signatures: Detail exact wordmarks, typography, overlapping geometric shapes (e.g. interlocking loops vs text wordmark), border radius, padding, and layout structure.
4. Color & Contrast Styling: Detail exact colors, contrast levels, and shadow/elevation effects.
5. Compliance & Style Classification: Classify whether this image represents an active/compliant style or an outdated/deprecated legacy pattern.
-Guideline for Google Pay (G Pay) compliance status:*
- The CURRENT COMPLIANT standard is the multi-colored interlocking loops design (four colored curved segments in blue, red, yellow, green forming a stylized double-loop G/Pay shape).
- Any logo showing the 'Google Pay' or 'G Pay' wordmark (where 'Google' is multi-colored and 'Pay' is grey/black/white) is an OUTDATED/LEGACY pattern."""

        resp = client.models.generate_content(
            model=GEMINI_ORCHESTRATOR_MODEL,
            contents=[image_part, forensic_analysis_prompt],
            config=types.GenerateContentConfig(temperature=0.0)
        )
        reference_image_description = resp.text
        print(f"Forensic Reference Image Analysis:\n{reference_image_description}\n{'='*50}")

    # Dynamically query all unique tags from AlloyDB visual_assets
    available_tags = []
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"SELECT DISTINCT tag FROM {DB_SCHEMA}.visual_assets, unnest(vision_tags) tag"
            )
            available_tags = [r["tag"] for r in rows if r["tag"]]
            print(f"Loaded {len(available_tags)} unique tag vocabulary terms from database.")
    except Exception as e:
        print(f"Warning: Could not fetch unique tags from DB ({e}). Using static fallback.")

    print("1. Translating goal into Audit Configuration...")
    audit_config = generate_audit_config(user_goal, reference_image_description, available_tags)
    print("Generated Configuration:\n", json.dumps(audit_config, indent=2))
    return audit_config

async def run_full_test_bench_pipeline_execution(audit_config: dict, reference_image_path: Optional[str] = None, quick_mode: bool = False) -> pd.DataFrame:
    """Stage 2, 3, 3.5, 4, 5: Executes Retrieval, Semantic Reranking, Inference, and Calibration scorecards."""
    pipeline_start_time = time.time()
    telemetry = {}

    t0 = time.time()
    print("\n2. Running Parallel 3-Arm Hybrid Search with RRF...")
    search_results = await run_hybrid_search({}, audit_config, reference_image_path)
    telemetry["Stage 2 (3-Arm RRF Retrieval)"] = f"{time.time() - t0:.2f}s | {len(search_results)} candidates retrieved"
    print(f"Found {len(search_results)} candidates.")

    t0 = time.time()
    print("\n3. Running Drop-off Analysis (Kneedle + Volatility)...")
    df_results = pd.DataFrame(search_results)
    df_high, df_edge, df_low = detect_dropoff_flawless(df_results)
    telemetry["Stage 3 (Drop-off Segmentation)"] = f"{time.time() - t0:.2f}s | High: {len(df_high)}, Borderline: {len(df_edge)}, Low (Filtered): {len(df_low)}"
    print(f"Rough Candidates -> High: {len(df_high)} | Borderline: {len(df_edge)} | Low: {len(df_low)}")

    t0 = time.time()
    print("\n3.5. Running Semantic Reranking and Fine Filtering...")
    df_high_sem, df_edge_sem, df_low_sem = await run_semantic_reranking_and_filter(df_high, df_edge, audit_config)
    telemetry["Stage 3.5 (Semantic Reranking)"] = f"{time.time() - t0:.2f}s | High: {len(df_high_sem)}, Borderline: {len(df_edge_sem)}"
    print(f"Semantic Candidates -> High: {len(df_high_sem)} | Borderline: {len(df_edge_sem)} | Low (Discarded): {len(df_low_sem)}, Low (Demoted): {len(df_low_sem)}")

    # Save full results globally for evaluation recall calculation
    globals()["df_results_full"] = df_results

    # Apply Quick Mode vs Smart Scan selection
    if quick_mode:
        print("\n Quick Mode enabled: Selecting top 30 High confidence and top 30 Borderline for visual inference.")
        candidates_high = df_high_sem.head(30) if not df_high_sem.empty else pd.DataFrame()
        candidates_edge = df_edge_sem.head(30) if not df_edge_sem.empty else pd.DataFrame()
    else:
        print("\n Smart Scan enabled: Selecting all High confidence and all Borderline for full visual inference.")
        candidates_high = df_high_sem
        candidates_edge = df_edge_sem

    t0 = time.time()
    print("\n4. Running Parallel LLM Audit Inference...")
    results_df = await run_llm_inference_on_dropoff_results(candidates_high, candidates_edge, audit_config, reference_image_path=reference_image_path)
    inf_duration = time.time() - t0
    throughput = len(results_df) / inf_duration if inf_duration > 0 else 0
    telemetry["Stage 4 (Parallel Visual Inference)"] = f"{inf_duration:.2f}s | Throughput: {throughput:.2f} images/sec"
    print(f"LLM Results: {len(results_df)} assets audited.")

    t0 = time.time()
    print("\n5. Generating Calibration Summary...")
    summary = await generate_ai_audit_summary(results_df, audit_config)
    telemetry["Stage 5 (Executive Summary Generation)"] = f"{time.time() - t0:.2f}s"

    total_time = time.time() - pipeline_start_time

    # Compute Observability Stats
    error_count = results_df["error"].notna().sum() if (not results_df.empty and "error" in results_df.columns) else 0
    high_conf = (results_df["match_confidence"] >= 95).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0
    borderline_conf = ((results_df["match_confidence"] >= 70) & (results_df["match_confidence"] < 95)).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0
    low_conf = (results_df["match_confidence"] < 70).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0

    try:
        from IPython.display import display, Markdown, HTML

        telemetry_md = f"""
### 📊 Enterprise Observability & Telemetry Scorecard
| Stage / Metric | Value / Duration | Status |
|:--- |:--- |:---|
| **Total Pipeline Execution Time** | **{total_time:.2f}s** | 🟢 Optimal Throughput |
| **Parallel Inference Speed** | **{throughput:.2f} images/sec** | Concurrency: 15 Workers |
| **Stage 2: 3-Arm RRF Retrieval** | {telemetry.get('Stage 2 (3-Arm RRF Retrieval)', 'N/A')} | HNSW + FTS + Tag Boost |
| **Stage 3: Kneedle Segmentation** | {telemetry.get('Stage 3 (Drop-off Segmentation)', 'N/A')} | Noise Tail Truncated |
| **Stage 3.5: Semantic Reranking** | {telemetry.get('Stage 3.5 (Semantic Reranking)', 'N/A')} | LLM Text Cross-Encoder |
| **Stage 4: Vision Inference** | {telemetry.get('Stage 4 (Parallel Visual Inference)', 'N/A')} | 0 False Positive Verdict Guarantee |
| **Confidence Band Distribution** | High (>95%): **{high_conf}** || Borderline (70-95%): **{borderline_conf}** || Low (<70%): **{low_conf}** || Error Count: **{error_count}** |
"""
        display(Markdown(telemetry_md))
        display(Markdown(summary))

        if not results_df.empty:
            display(Markdown("### Detailed Audit Results"))
            display_df = results_df.copy()
            display_df["Visual Preview"] = display_df["gcs_raw_path"].apply(
                lambda x: f'<img src="{get_gcs_image_base64(x)}" width="150" />' if get_gcs_image_base64(x) else '[No Preview]'
            )
            display_df["Page Link"] = display_df["page_url"].apply(
                lambda x: f'<a href="{x}" target="_blank">{x}</a>' if x else '[No Page Link]'
            )
            cols = ["Visual Preview", "matches_criteria", "relevance_score", "gcs_raw_path", "Page Link", "match_rationale"]
            col_labels = ["Visual Preview", "Matches Criteria", "Search Similarity", "GCS Path", "Page Location URL", "Rationale"]

            if "gemini_description" in display_df.columns:
                cols.append("gemini_description")
                col_labels.append("Image Description")

            display_df = display_df[cols]
            display_df.columns = col_labels

            display(HTML(display_df.to_html(escape=False, index=False)))
    except (ImportError, ModuleNotFoundError):
        print("\n=== TELEMETRY SCORECARD ===")
        for k, v in telemetry.items():
            print(f"  {k}: {v}")
        print(f"  Total Time: {total_time:.2f}s | Errors: {error_count}")
        print("\n=== CALIBRATION SUMMARY ===")
        print(summary)

    return results_df

# Bypassed original monolithic E2E pipeline name to map to partitioned runners
async def run_full_test_bench_pipeline(user_goal: str, reference_image_path: Optional[str] = None, quick_mode: bool = False) -> pd.DataFrame:
    """Wrapper to maintain backwards compatibility for existing cells."""
    config = await generate_audit_config_only(user_goal, reference_image_path)
    return await run_full_test_bench_pipeline_execution(config, reference_image_path, quick_mode=quick_mode)


## E2E Execution & Telemetry Helpers

This cell defines the core pipeline orchestrators: `generate_audit_config_only` (Stage 1 Config) and `run_full_test_bench_pipeline_execution` (Stage 2 E2E). It coordinates retrieval, segmentation, cross-encoder reranking, and visual LLM inference, while logging telemetry and formatted scorecard widgets.

In [ ]:
# TEST RUN (Stage 1): Generate Audit Rules & Configuration
# Run this cell to upload your reference image and generate the rule configuration.
import nest_asyncio
nest_asyncio.apply()

# Dynamic Reference Image Detector (Triggered via Colab Interactive Upload)
reference_image_path = None
try:
    from google.colab import files
    print("[OPTIONAL] Upload a reference image for comparative compliance audit:")
    uploaded = files.upload()
    if uploaded:
        reference_image_path = list(uploaded.keys())[0]
        print(f" Reference image uploaded: {reference_image_path}")
    else:
        print(" No reference image uploaded. Running text-only audit goal.")
except Exception as e:
    reference_image_path = None

# Enterprise Audit Goal (Modify this as needed)
TEST_AUDIT_GOAL = "Find all pages with this exact image"

# Step 1: Generate configuration rules
audit_config_global = None
try:
    audit_config_global = await generate_audit_config_only(
        user_goal=TEST_AUDIT_GOAL,
        reference_image_path=reference_image_path
    )
    print("💡 TIP: You can inspect and tweak 'audit_config_global' directly in the cell below before running Stage 2.")
except Exception as e:
    print(f"Error generating audit configuration: {e}")


In [ ]:
# TEST RUN (Stage 2): Execute Visual Search & Parallel Audit
# Run this cell to execute retrieval, segmentation, and LLM inference.
# You can uncomment and modify rules below to calibrate config before executing.

if 'audit_config_global' in globals() and audit_config_global is not None:
    # OPTIONAL CALIBRATION TUNING:
    # If the generated AI rules were slightly off, you can uncomment and edit them here:
    # audit_config_global["inclusion_criteria"] = [
    #     "The image must contain the legacy Google Pay logo featuring the interlocking loops design."
    # ]
    # audit_config_global["exclusion_criteria"] = [
    #     "Exclude images containing the current compliant Google Pay GPay wordmark button logo."
    # ]

    df_results_global = None
    try:
        df_results_global = await run_full_test_bench_pipeline_execution(
            audit_config=audit_config_global,
            reference_image_path=reference_image_path
        )
    except Exception as e:
        print(f"Error executing test bench pipeline: {e}")
else:
    print(" Please run Stage 1 cell first to generate 'audit_config_global'.")
